##L05: Silver, step 1: conform, cast and deduplicate

###Step 1. Rename and cast

In [0]:
from pyspark.sql import functions as F

b = spark.table("da.bronze.yellow_trips")
bronze_cols = set(b.columns)  # one Analyze RPC, reused everywhere below

# 1. Rename cryptic TLC names to snake_case — all at once, no loop
rename = {"VendorID": "vendor_id", "tpep_pickup_datetime": "pickup_ts",
          "tpep_dropoff_datetime": "dropoff_ts", "RatecodeID": "rate_code_id",
          "PULocationID": "pu_location_id", "DOLocationID": "do_location_id",
          "Airport_fee": "airport_fee"}
df = b.withColumnsRenamed({old: new for old, new in rename.items() if old in bronze_cols})

# column names after the rename step, tracked locally — no more Analyze RPCs
current_cols = {rename.get(c, c) for c in bronze_cols}

# 2. Cast money columns to decimal(10,2); create missing ones as NULL columns
money = ["fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
         "improvement_surcharge", "total_amount", "congestion_surcharge",
         "airport_fee", "cbd_congestion_fee"]
df = df.withColumns({c: (F.col(c) if c in current_cols else F.lit(None)).cast("decimal(10,2)")
                     for c in money})

# 3. Cast the rest and normalize store_and_fwd_flag — one flat projection
s = (df.withColumns({
        "vendor_id": F.col("vendor_id").cast("int"),
        "pickup_ts": F.col("pickup_ts").cast("timestamp"),
        "dropoff_ts": F.col("dropoff_ts").cast("timestamp"),
        "passenger_count": F.col("passenger_count").cast("int"),
        "rate_code_id": F.col("rate_code_id").cast("int"),
        "pu_location_id": F.col("pu_location_id").cast("int"),
        "do_location_id": F.col("do_location_id").cast("int"),
        "payment_type": F.col("payment_type").cast("int"),
        "trip_distance": F.col("trip_distance").cast("double"),
        "store_and_fwd_flag": F.upper(F.trim(F.col("store_and_fwd_flag")))}))

s.printSchema()  # forces analysis now — catches any column typo and shows the conformed schema

For this particular step, SQL is genuinely simpler. 
SQL Equivalent — one flat SELECT: aliases rename, CAST conforms types, and a missing column is just CAST(NULL AS ...)

Where python is better in loading pipeline:
-  Schemas that drift or vary — the if old in df.columns / lit(None) logic lets one pipeline handle taxi files where cbd_congestion_fee or Airport_fee may or may not exist. With SQL, a missing column breaks the whole query.
- Config-driven pipelines — when the rename map or money-column list comes from a config file or is shared across tables, Python loops over data structures; SQL makes you hardcode every column.
- Composability — s flows into the deduplicate step and the silver write as a DataFrame, and intermediate schemas can be inspected/tested at each stage. SQL would need CTEs or views to chain the same logic.



In [0]:
%sql
SELECT
  CAST(VendorID AS INT)                          AS vendor_id,
  CAST(tpep_pickup_datetime AS TIMESTAMP)         AS pickup_ts,
  CAST(tpep_dropoff_datetime AS TIMESTAMP)        AS dropoff_ts,
  CAST(passenger_count AS INT)                    AS passenger_count,
  CAST(trip_distance AS DOUBLE)                   AS trip_distance,
  CAST(RatecodeID AS INT)                         AS rate_code_id,
  UPPER(TRIM(store_and_fwd_flag))                 AS store_and_fwd_flag,
  CAST(PULocationID AS INT)                       AS pu_location_id,
  CAST(DOLocationID AS INT)                       AS do_location_id,
  CAST(payment_type AS INT)                       AS payment_type,
  CAST(fare_amount AS DECIMAL(10,2))              AS fare_amount,
  CAST(extra AS DECIMAL(10,2))                    AS extra,
  CAST(mta_tax AS DECIMAL(10,2))                  AS mta_tax,
  CAST(tip_amount AS DECIMAL(10,2))               AS tip_amount,
  CAST(tolls_amount AS DECIMAL(10,2))            AS tolls_amount,
  CAST(improvement_surcharge AS DECIMAL(10,2))    AS improvement_surcharge,
  CAST(total_amount AS DECIMAL(10,2))             AS total_amount,
  CAST(congestion_surcharge AS DECIMAL(10,2))     AS congestion_surcharge,
  CAST(Airport_fee AS DECIMAL(10,2))              AS airport_fee,
  CAST(NULL AS DECIMAL(10,2))                     AS cbd_congestion_fee,
  source_file,
  ingested_at
FROM da.bronze.yellow_trips

###Step 2. Add derived columns and a stable key
trip_id is a hash of the natural attributes, so reruns give the same key.

In [0]:
s = (s.withColumn("pickup_date", F.to_date("pickup_ts"))
       .withColumn("trip_minutes", (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60.0)
       .withColumn("trip_id", F.sha2(F.concat_ws("|", F.col("vendor_id").cast("string"),
                   F.col("pickup_ts").cast("string"), F.col("dropoff_ts").cast("string"),
                   F.col("pu_location_id").cast("string"), F.col("do_location_id").cast("string"),
                   F.col("fare_amount").cast("string"), F.col("total_amount").cast("string")), 256)))

   
SQL Equivalent — SQL can't reference `s` across cells, so the step-1 conform is wrapped in a `conformed` CTE and the derived columns + hash key are built on top. Same SHA-256 over the same cast attributes, so the keys match the Python version exactly.

In [0]:
%sql
WITH conformed AS (
  -- Step 1: conform and cast (same as the previous SQL cell)
  SELECT
    CAST(VendorID AS INT)                        AS vendor_id,
    CAST(tpep_pickup_datetime AS TIMESTAMP)      AS pickup_ts,
    CAST(tpep_dropoff_datetime AS TIMESTAMP)     AS dropoff_ts,
    CAST(passenger_count AS INT)                 AS passenger_count,
    CAST(trip_distance AS DOUBLE)                AS trip_distance,
    CAST(RatecodeID AS INT)                      AS rate_code_id,
    UPPER(TRIM(store_and_fwd_flag))              AS store_and_fwd_flag,
    CAST(PULocationID AS INT)                    AS pu_location_id,
    CAST(DOLocationID AS INT)                    AS do_location_id,
    CAST(payment_type AS INT)                    AS payment_type,
    CAST(fare_amount AS DECIMAL(10,2))           AS fare_amount,
    CAST(extra AS DECIMAL(10,2))                 AS extra,
    CAST(mta_tax AS DECIMAL(10,2))               AS mta_tax,
    CAST(tip_amount AS DECIMAL(10,2))            AS tip_amount,
    CAST(tolls_amount AS DECIMAL(10,2))          AS tolls_amount,
    CAST(improvement_surcharge AS DECIMAL(10,2)) AS improvement_surcharge,
    CAST(total_amount AS DECIMAL(10,2))          AS total_amount,
    CAST(congestion_surcharge AS DECIMAL(10,2)) AS congestion_surcharge,
    CAST(Airport_fee AS DECIMAL(10,2))           AS airport_fee,
    CAST(NULL AS DECIMAL(10,2))                  AS cbd_congestion_fee,
    source_file,
    ingested_at
  FROM da.bronze.yellow_trips
)
SELECT
  conformed.*,
  to_date(pickup_ts)                                             AS pickup_date,
  (unix_timestamp(dropoff_ts) - unix_timestamp(pickup_ts)) / 60.0 AS trip_minutes,
  sha2(concat_ws('|',
        CAST(vendor_id      AS STRING),
        CAST(pickup_ts      AS STRING),
        CAST(dropoff_ts     AS STRING),
        CAST(pu_location_id  AS STRING),
        CAST(do_location_id AS STRING),
        CAST(fare_amount    AS STRING),
        CAST(total_amount   AS STRING)), 256)                     AS trip_id
FROM conformed

###Step 3. Deduplicate and report
Count rows before and after. Keep this number for your README.

In [0]:
before = s.count()
s = s.dropDuplicates(["trip_id"])
after = s.count()
print("before", before, "after", after, "removed", before - after)

   
SQL Equivalent — the report collapses to one aggregate: total rows, distinct `trip_id`s, and the difference. To materialize the deduped table like `dropDuplicates`, use `ROW_NUMBER() OVER (PARTITION BY trip_id ORDER BY ingested_at) = 1` — deterministic, unlike `dropDuplicates` which keeps an arbitrary row.

In [0]:
%sql
WITH conformed AS (
  -- Step 1: conform and cast (same as the previous SQL cell)
  SELECT
    CAST(VendorID AS INT)                        AS vendor_id,
    CAST(tpep_pickup_datetime AS TIMESTAMP)      AS pickup_ts,
    CAST(tpep_dropoff_datetime AS TIMESTAMP)     AS dropoff_ts,
    CAST(passenger_count AS INT)                 AS passenger_count,
    CAST(trip_distance AS DOUBLE)                AS trip_distance,
    CAST(RatecodeID AS INT)                      AS rate_code_id,
    UPPER(TRIM(store_and_fwd_flag))              AS store_and_fwd_flag,
    CAST(PULocationID AS INT)                    AS pu_location_id,
    CAST(DOLocationID AS INT)                    AS do_location_id,
    CAST(payment_type AS INT)                    AS payment_type,
    CAST(fare_amount AS DECIMAL(10,2))           AS fare_amount,
    CAST(extra AS DECIMAL(10,2))                 AS extra,
    CAST(mta_tax AS DECIMAL(10,2))               AS mta_tax,
    CAST(tip_amount AS DECIMAL(10,2))            AS tip_amount,
    CAST(tolls_amount AS DECIMAL(10,2))          AS tolls_amount,
    CAST(improvement_surcharge AS DECIMAL(10,2)) AS improvement_surcharge,
    CAST(total_amount AS DECIMAL(10,2))          AS total_amount,
    CAST(congestion_surcharge AS DECIMAL(10,2)) AS congestion_surcharge,
    CAST(Airport_fee AS DECIMAL(10,2))           AS airport_fee,
    CAST(NULL AS DECIMAL(10,2))                  AS cbd_congestion_fee,
    source_file,
    ingested_at
  FROM da.bronze.yellow_trips
),
keyed AS (
  -- Step 2: the stable hash key (same as the previous SQL cell)
  SELECT
    sha2(concat_ws('|',
          CAST(vendor_id      AS STRING),
          CAST(pickup_ts      AS STRING),
          CAST(dropoff_ts     AS STRING),
          CAST(pu_location_id  AS STRING),
          CAST(do_location_id AS STRING),
          CAST(fare_amount    AS STRING),
          CAST(total_amount   AS STRING)), 256) AS trip_id
  FROM conformed
)
SELECT
  count(*)                        AS before_rows,
  count(DISTINCT trip_id)         AS after_rows,
  count(*) - count(DISTINCT trip_id) AS removed
FROM keyed

###Step 4. Write the table

In [0]:
(s.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("da.silver.yellow_trips_conformed"))